In [ ]:
# /home/ec2-user/exp/step7_tracks_tmux_latest/run_step7_full_failure_salvage_tmux.sh

# Structural Connectome B

EC2-native post-Eddy pipeline notebook for Steps 5 through 7.

This notebook owns:
- the portable pipeline-status table cell
- T1 preprocessing (Step 5)
- post-Eddy stage snapshots (Step 6)
- BBR and connectome staging (Step 7)


## Pipeline Inventory Table

Run this cell first to see the EC2 pipeline completeness snapshot across the intermediate stages. It uses the current contents of `~/exp/data/derivatives` and helps show which stage outputs are still missing.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import sys

PROJECT_ROOT = Path.home() / "exp"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for candidate in [
    Path.home() / "bin",
    Path.home() / "mrtrix3" / "bin",
    Path.home() / "fsl" / "bin",
    Path.home() / "fsl" / "share" / "fsl" / "bin",
]:
    if candidate.exists() and str(candidate) not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = f"{candidate}{os.pathsep}" + os.environ.get("PATH", "")

os.environ.setdefault("FSLDIR", str(Path.home() / "fsl"))

from connectome_pipeline.pipeline_paths import resolve_pipeline_paths

paths = resolve_pipeline_paths(create_layout=True)
print(paths.format_summary())

from connectome_pipeline import connectome_run_control
connectome_run_control = importlib.reload(connectome_run_control)

DRY_RUN = True
EXECUTE = False
FORCE_INVALID_ONLY = True
INCLUDE = []
EXCLUDE = []
BACKUP_BEFORE_REPLACE = True
VALIDATION_GATE = True
INCLUDE_WARN_QC = False

RUN_CONTROL = connectome_run_control.RunControlConfig.from_values(
    dry_run=DRY_RUN,
    execute=EXECUTE,
    force_invalid_only=FORCE_INVALID_ONLY,
    include=INCLUDE,
    exclude=EXCLUDE,
    backup_before_replace=BACKUP_BEFORE_REPLACE,
    validation_gate=VALIDATION_GATE,
    include_warn=INCLUDE_WARN_QC,
)
connectome_run_control.print_run_control(RUN_CONTROL)


def module_status(*names):
    return {name: importlib.util.find_spec(name) is not None for name in names}


def visible_children(path: Path, limit: int = 10):
    if not path.exists():
        return []
    return [p.name for p in sorted(path.iterdir()) if not p.name.startswith('.')][:limit]


## 10. Canary Selection From Current QC

This picks a small canary panel from current matrix QC: severe failures, moderate failures, and controls. It is a planning list, not an automatic rerun.

In [ ]:
import importlib
from connectome_pipeline import pipeline_status

pipeline_status = importlib.reload(pipeline_status)

status_location = "ec2"  # change to "s3" to read counts from the bucket

all_stage_status = pipeline_status.display_all_stage_group_status(
    paths.deriv_root,
    cohort_dti_csv=paths.cohort_dti_csv,
    aal_mni=paths.aal_mni,
    location=status_location,
    s3_root="s3://sabeesh/exp",
)
all_stage_status


## Step 5: T1 Anatomy Preprocessing

This step becomes runnable once raw T1 data is pushed into `~/exp/data/Images/mri`.


In [ ]:
import importlib
import os

from connectome_pipeline import pipeline_status
import t1_pipeline

pipeline_status = importlib.reload(pipeline_status)
t1_pipeline = importlib.reload(t1_pipeline)

T1_CFG = {
    "raw_t1_root": paths.raw_t1_root,
    "t1_anat_dir": paths.deriv_root / "t1_anat",
    "t1_fast_dir": paths.deriv_root / "t1_fast",
    "modality_names": set(t1_pipeline.DEFAULT_T1_MODALITY_NAMES),
    "force": False,
    # "max_workers": min(2, max(1, (os.cpu_count() or 4) // 2)),
    "max_workers": max(2, max(1, (os.cpu_count() or 4) // 2)),
}

raw_t1_children = visible_children(paths.raw_t1_root)
eddy_subjects = t1_pipeline.find_eddy_subjects(paths.deriv_root / "eddy")
selected_t1_subjects = eddy_subjects
print(f"raw_t1_root exists       : {paths.raw_t1_root.exists()}")
print(f"raw_t1_root sample       : {raw_t1_children}")
print(f"Eddy subjects with DWI   : {len(eddy_subjects)}")
print(f"T1 subjects scheduled    : {len(selected_t1_subjects)}")
print(f"Example subjects         : {selected_t1_subjects[:10]}")
if not raw_t1_children:
    print("Step 5 is gated until raw T1 data is pushed into ~/exp/data/Images/mri.")
print()
pipeline_status.print_t1_status(paths.deriv_root)


In [ ]:
if not visible_children(paths.raw_t1_root):
    print("Skipping Step 5 run: no raw T1 inputs are present yet.")
else:
    t1_result = t1_pipeline.run_t1_preprocessing(
        raw_t1_root=T1_CFG["raw_t1_root"],
        t1_anat_dir=T1_CFG["t1_anat_dir"],
        t1_fast_dir=T1_CFG["t1_fast_dir"],
        modality_names=T1_CFG["modality_names"],
        select_subjects=selected_t1_subjects,
        force=T1_CFG["force"],
        max_workers=T1_CFG["max_workers"],
    )
    t1_status_after = pipeline_status.print_t1_status(paths.deriv_root)
    t1_result


## Step 6: Stage Snapshot and Bias-Correction Readiness


In [ ]:
import importlib.util

raw_dwi_ready = bool(visible_children(paths.raw_dwi_root)) and bool(module_status("dwi_convert")["dwi_convert"])
raw_dwi_root_for_summary = paths.raw_dwi_root if raw_dwi_ready else None

step6_summary = pipeline_status.print_step6_summary(
    paths.deriv_root,
    raw_dwi_root=raw_dwi_root_for_summary,
)
print()
pipeline_status.print_t1_status(paths.deriv_root)
print()
pipeline_status.print_biascorr_status(paths.deriv_root)
print()
pipeline_status.print_bbr_status(paths.deriv_root)
print()
print(f"bias_correction.py available : {module_status('bias_correction')['bias_correction']}")
if not module_status('bias_correction')['bias_correction']:
    print('Step 6a/6c remain gated until bias_correction.py is copied into ~/exp.')


In [ ]:
if not module_status('bias_correction')['bias_correction']:
    print('Skipping Step 6a run: bias_correction.py is not present in ~/exp.')
else:
    import bias_correction
    import importlib

    bias_correction = importlib.reload(bias_correction)
    BIAS_STAGE_ROOT = Path.home() / 'staging' / 'biascorr_stage'
    bias_result = bias_correction.run_bias_correction(
        deriv=paths.deriv_root,
        stage_root=BIAS_STAGE_ROOT,
        # processes=max(1, (os.cpu_count() or 4) // 3),
        processes=48,
        threads_per_job=2,
        force=False,
        run_mode='thread',
        select_series=[],
        debug=False,
        allow_fsl_fallback=False,
    )
    bias_status_after = pipeline_status.print_biascorr_status(paths.deriv_root)
    bias_result


## Step 6d: T1 to DWI BBR Bridge


In [ ]:
import bbr_bridge

bbr_bridge = importlib.reload(bbr_bridge)

BBR_TARGET_MODE = 'repair'  # repair | full | manual
BBR_REPAIR_TARGET_FILE = Path('/home/ec2-user/exp/step7_tracks_tmux_latest/tracks_repair_geometry_rerun_bbr_prep.txt')
BBR_MANUAL_INCLUDE = []

def _read_bbr_subject_file(path):
    path = Path(path)
    if not path.exists():
        return []
    return [line.strip() for line in path.read_text().splitlines() if line.strip() and not line.lstrip().startswith('#')]

def resolve_bbr_ui_targets():
    manual = list(dict.fromkeys(BBR_MANUAL_INCLUDE or []))
    if manual:
        return manual, 'manual'
    if BBR_TARGET_MODE == 'repair':
        return list(dict.fromkeys(_read_bbr_subject_file(BBR_REPAIR_TARGET_FILE))), 'repair'
    if BBR_TARGET_MODE == 'full':
        return [], 'full'
    raise ValueError("BBR_TARGET_MODE must be 'repair', 'full', or 'manual'")

_bbr_selected, _bbr_mode = resolve_bbr_ui_targets()

BBR_CFG = {
    "deriv": paths.deriv_root,
    "force": bool(_bbr_selected),
    "stage_root": Path.home() / 'staging' / 'bbr_bridge_stage',
    "max_workers": 2,
    "select_sids": _bbr_selected,
    "overwrite_logs": True,
}

print('BBR target mode:', _bbr_mode, '| selected:', len(_bbr_selected), '| force:', BBR_CFG['force'])
if _bbr_selected:
    print('BBR repair targets:', ', '.join(_bbr_selected))

pipeline_status.print_bbr_status(paths.deriv_root)


In [ ]:
bbr_result = bbr_bridge.run_bbr_for_missing(
    deriv=BBR_CFG['deriv'],
    force=BBR_CFG['force'],
    stage_root=BBR_CFG['stage_root'],
    max_workers=BBR_CFG['max_workers'],
    select_sids=BBR_CFG['select_sids'],
    overwrite_logs=BBR_CFG['overwrite_logs'],
)
bbr_status_after = pipeline_status.print_bbr_status(paths.deriv_root)
bbr_result


## Step 7: Connectome Generation


In [ ]:
from pathlib import Path
import csv
import importlib
from connectome_pipeline import connectome_run_control
from connectome_pipeline import connectome_step7

STEP7_BATCH_SIZE = 48
STEP7_SCAN_WORKERS = 0
STEP7_TCK_THREADS = 1
STEP7_OMP_5TT_THREADS = 1
STEP7_FORCE = bool(RUN_CONTROL.execute and not RUN_CONTROL.force_invalid_only)
STEP7_TRACK_SEED_MODE = 'gmwmi'  # gmwmi | dynamic
STEP7_TARGET_MODE = 'repair'  # repair | full
STEP7_REPAIR_TARGET_DIR = Path('/home/ec2-user/exp/step7_tracks_tmux_latest')

# Spatial-contract controls. Defaults preserve the current production behavior.
# Switch these only for canary/debug cells after QC evidence supports the route.
STEP7_T1_B0_ROUTE = 'current_bbr'  # current_bbr | reference_flirt_dof6
STEP7_FIVE_TT_ROUTE = 'current_transform'  # current_transform | reference_b0_t1
STEP7_AAL_TRANSFORM_ROUTE = 'current_direct'  # current_direct | supervisor_two_step
STEP7_ASSIGNMENT_VARIANT = 'radial4'  # radial4 | radial8 | forward40 | forward80 | none | legacy
STEP7_BACKUP_BEFORE_REPLACE = RUN_CONTROL.backup_before_replace
STEP7_VALIDATION_GATE = RUN_CONTROL.validation_gate


def make_step7_cfg(*, force=None, track_seed_mode=None):
    global connectome_step7, connectome_run_control
    connectome_run_control = importlib.reload(connectome_run_control)
    connectome_step7 = importlib.reload(connectome_step7)
    return connectome_step7.Step7Config(
        deriv_root=paths.deriv_root,
        bias_root=paths.deriv_root / 'biascorr_1',
        bbr_root=paths.deriv_root / 'dwi_t1_bbr',
        aal_mni=paths.aal_mni,
        batch_size=STEP7_BATCH_SIZE,
        scan_workers=STEP7_SCAN_WORKERS,
        tck_threads=STEP7_TCK_THREADS,
        omp_5tt_threads=STEP7_OMP_5TT_THREADS,
        track_seed_mode=track_seed_mode or STEP7_TRACK_SEED_MODE,
        force=STEP7_FORCE if force is None else bool(force),
        assignment_variant=STEP7_ASSIGNMENT_VARIANT,
        t1_b0_route=STEP7_T1_B0_ROUTE,
        five_tt_route=STEP7_FIVE_TT_ROUTE,
        aal_transform_route=STEP7_AAL_TRANSFORM_ROUTE,
        backup_before_replace=STEP7_BACKUP_BEFORE_REPLACE,
        validation_gate=STEP7_VALIDATION_GATE,
        analysis_gate_csv=RUN_CONTROL.analysis_gate_csv,
    )


def _step7_read_subject_file(path):
    path = Path(path)
    if not path.exists():
        return []
    return [
        line.strip()
        for line in path.read_text().splitlines()
        if line.strip() and not line.lstrip().startswith('#')
    ]


def _step7_unique(items):
    seen = set()
    out = []
    for item in items or []:
        if item and item not in seen:
            seen.add(item)
            out.append(item)
    return out


def _step7_latest_summary_rows(cfg, subjects):
    latest = {}
    targets = set(subjects or [])
    summary_csv = Path(cfg.summary_csv)
    if not targets or not summary_csv.exists():
        return latest
    with summary_csv.open(newline='', encoding='utf-8', errors='replace') as handle:
        for row in csv.DictReader(handle):
            sid = (row.get('sid') or '').strip()
            stage = (row.get('stage') or '').strip()
            if sid in targets and stage in {'prep', 'fod', 'tracks', 'post'}:
                latest[(sid, stage)] = row
    return latest


def _step7_summary_status(latest, sid, stage):
    return (latest.get((sid, stage), {}).get('status') or '').strip()


def _step7_fod_pending_targets(latest, subjects):
    pending = []
    blocked = {}
    for sid in subjects:
        prep_status = _step7_summary_status(latest, sid, 'prep')
        fod_status = _step7_summary_status(latest, sid, 'fod')
        if prep_status and prep_status != 'ok':
            blocked[sid] = f'prep_{prep_status}'
            continue
        if fod_status != 'ok':
            pending.append(sid)
    return pending, blocked


def _step7_tracks_ready_targets(latest, subjects):
    return [sid for sid in subjects if _step7_summary_status(latest, sid, 'fod') == 'ok']


def get_step7_ui_repair_targets(cfg=None):
    cfg = cfg or STEP7_CFG
    geometry = _step7_read_subject_file(STEP7_REPAIR_TARGET_DIR / 'tracks_repair_geometry_force_prep_fod.txt')
    dynamic = _step7_read_subject_file(STEP7_REPAIR_TARGET_DIR / 'tracks_repair_dynamic_seed.txt')
    other = _step7_read_subject_file(STEP7_REPAIR_TARGET_DIR / 'tracks_repair_other_review.txt')
    latest = _step7_latest_summary_rows(cfg, geometry + dynamic + other)
    geometry_fod_pending, geometry_fod_blocked = _step7_fod_pending_targets(latest, geometry)
    geometry_tracks_ready = _step7_tracks_ready_targets(latest, geometry)
    # Keep all dynamic-lane subjects eligible here; the tracks stage itself performs strict FOD/ACT preflight.
    dynamic_tracks_ready = list(dynamic)
    bad_fod_dynamic = [sid for sid in dynamic if _step7_summary_status(latest, sid, 'fod') not in {'', 'ok'}]
    bad_fod_details = {sid: _step7_summary_status(latest, sid, 'fod') for sid in bad_fod_dynamic}
    return {
        'geometry': geometry,
        'dynamic': dynamic,
        'other': other,
        'geometry_fod_pending': geometry_fod_pending,
        'geometry_fod_blocked': geometry_fod_blocked,
        'geometry_tracks_ready': geometry_tracks_ready,
        'dynamic_tracks_ready': dynamic_tracks_ready,
        'bad_fod_dynamic': bad_fod_dynamic,
        'bad_fod_details': bad_fod_details,
        'all': _step7_unique(geometry + dynamic + other),
    }


def _resolve_step7_ui_lanes(stage, include, *, force=None, track_seed_mode=None):
    stage = str(stage).strip().lower()
    manual_include = _step7_unique(include or [])
    if manual_include or STEP7_TARGET_MODE == 'full':
        return [
            {
                'label': 'manual' if manual_include else 'full',
                'stage': stage,
                'include': manual_include or None,
                'force': STEP7_FORCE if force is None else bool(force),
                'track_seed_mode': track_seed_mode or STEP7_TRACK_SEED_MODE,
            }
        ]

    targets = get_step7_ui_repair_targets(STEP7_CFG)
    lanes = []
    if stage == 'prep':
        if targets['geometry']:
            lanes.append({
                'label': 'repair_geometry_prep',
                'stage': 'prep',
                'include': targets['geometry'],
                'force': True if force is None else bool(force),
                'track_seed_mode': 'gmwmi',
            })
    elif stage == 'fod':
        if targets['geometry_fod_pending']:
            lanes.append({
                'label': 'repair_geometry_fod_pending',
                'stage': 'fod',
                'include': targets['geometry_fod_pending'],
                'force': True if force is None else bool(force),
                'track_seed_mode': 'gmwmi',
            })
        if targets['bad_fod_dynamic']:
            lanes.append({
                'label': 'repair_dynamic_bad_fod',
                'stage': 'fod',
                'include': targets['bad_fod_dynamic'],
                'force': True if force is None else bool(force),
                'track_seed_mode': 'dynamic',
            })
    elif stage == 'tracks':
        if targets['geometry_tracks_ready']:
            lanes.append({
                'label': 'repair_geometry_tracks_ready',
                'stage': 'tracks',
                'include': targets['geometry_tracks_ready'],
                'force': STEP7_FORCE if force is None else bool(force),
                'track_seed_mode': 'gmwmi',
            })
        if targets['dynamic_tracks_ready']:
            lanes.append({
                'label': 'repair_dynamic_tracks_ready',
                'stage': 'tracks',
                'include': targets['dynamic_tracks_ready'],
                'force': STEP7_FORCE if force is None else bool(force),
                'track_seed_mode': 'dynamic',
            })
    elif stage == 'post':
        if targets['all']:
            lanes.append({
                'label': 'repair_post',
                'stage': 'post',
                'include': targets['all'],
                'force': STEP7_FORCE if force is None else bool(force),
                'track_seed_mode': track_seed_mode or STEP7_TRACK_SEED_MODE,
            })
    else:
        lanes.append({
            'label': 'repair_auto',
            'stage': stage,
            'include': targets['all'] or None,
            'force': STEP7_FORCE if force is None else bool(force),
            'track_seed_mode': track_seed_mode or STEP7_TRACK_SEED_MODE,
        })
    return lanes


def run_step7_ui_stage(stage, include=None, exclude=None, *, force=None, track_seed_mode=None):
    global STEP7_CFG, step7_result, step7_status_after
    exclude = exclude or None
    STEP7_CFG = make_step7_cfg(force=force, track_seed_mode=track_seed_mode)
    lanes = _resolve_step7_ui_lanes(stage, include, force=force, track_seed_mode=track_seed_mode)
    if not lanes:
        print(f'No Step 7 targets resolved for stage={stage!r} in target_mode={STEP7_TARGET_MODE!r}.')
        step7_result = {'stage': stage, 'ok': True, 'lanes': []}
        step7_status_after = None
        return step7_result

    lane_results = []
    status_include = []
    for lane in lanes:
        lane_include = lane['include']
        if lane_include:
            status_include.extend(lane_include)
        STEP7_CFG = make_step7_cfg(force=lane['force'], track_seed_mode=lane['track_seed_mode'])
        target_count = 'all' if lane_include is None else len(lane_include)
        print(
            f"Running Step 7 lane: {lane['label']} | stage={lane['stage']} | "
            f"targets={target_count} | force={STEP7_CFG.force} | seed_mode={STEP7_CFG.track_seed_mode} | "
            f"aal_route={STEP7_CFG.aal_transform_route} | assignment={STEP7_CFG.assignment_variant}"
        )
        result = connectome_step7.run_step7_pipeline(
            STEP7_CFG,
            stage=lane['stage'],
            include=lane_include,
            exclude=exclude,
        )
        lane_results.append({**lane, 'result': result})

    status_include = _step7_unique(status_include) or None
    step7_status_after = connectome_step7.print_step7_status_quick(
        STEP7_CFG,
        include=status_include,
        exclude=exclude,
    )
    step7_result = {
        'stage': stage,
        'target_mode': STEP7_TARGET_MODE,
        'lanes': lane_results,
        'status_include': status_include,
    }
    return step7_result


STEP7_CFG = make_step7_cfg()

step7_quick = connectome_step7.print_step7_status_quick(STEP7_CFG)
print()
step7_ui_repair_targets = get_step7_ui_repair_targets(STEP7_CFG)
print(
    'Step 7 UI target mode:', STEP7_TARGET_MODE,
    '| geometry:', len(step7_ui_repair_targets['geometry']),
    '| geometry_fod_pending:', len(step7_ui_repair_targets['geometry_fod_pending']),
    '| geometry_tracks_ready:', len(step7_ui_repair_targets['geometry_tracks_ready']),
    '| dynamic:', len(step7_ui_repair_targets['dynamic']),
    '| dynamic_tracks_ready:', len(step7_ui_repair_targets['dynamic_tracks_ready']),
    '| bad_fod_dynamic:', len(step7_ui_repair_targets['bad_fod_dynamic']),
    '| other:', len(step7_ui_repair_targets['other']),
)
if step7_ui_repair_targets['geometry_fod_pending']:
    print('geometry_fod_pending targets:', ', '.join(step7_ui_repair_targets['geometry_fod_pending']))
if step7_ui_repair_targets['geometry_fod_blocked']:
    print('geometry_fod_blocked:', step7_ui_repair_targets['geometry_fod_blocked'])
if step7_ui_repair_targets['bad_fod_dynamic']:
    print('bad_fod_dynamic targets:', ', '.join(step7_ui_repair_targets['bad_fod_dynamic']))
print()
step7_group_status = connectome_step7.collect_step7_group_status(
    STEP7_CFG,
    cohort_dti_csv=paths.cohort_dti_csv,
    scope='full',
)
step7_group_status


### <font color = 'yellow'> Step 7: Connectome Generation : Prep


In [ ]:
STEP7_STAGE = 'prep'
STEP7_INCLUDE = []
STEP7_EXCLUDE = []

step7_result = run_step7_ui_stage(
    STEP7_STAGE,
    include=STEP7_INCLUDE,
    exclude=STEP7_EXCLUDE,
)
step7_result


### <font color = 'yellow'> Step 7: Connectome Generation : FOD


In [ ]:
STEP7_STAGE = 'fod'
STEP7_INCLUDE = []
STEP7_EXCLUDE = []

step7_result = run_step7_ui_stage(
    STEP7_STAGE,
    include=STEP7_INCLUDE,
    exclude=STEP7_EXCLUDE,
)
step7_result


### <font color = 'yellow'> Step 7: Connectome Generation : Tracks


In [ ]:
STEP7_STAGE = 'tracks'
STEP7_INCLUDE = []
STEP7_EXCLUDE = []

step7_result = run_step7_ui_stage(
    STEP7_STAGE,
    include=STEP7_INCLUDE,
    exclude=STEP7_EXCLUDE,
)
step7_result


## Pipeline Inventory Table

Run this cell first to see the EC2 pipeline completeness snapshot across the intermediate stages. It uses the current contents of `~/exp/data/derivatives` and helps show which stage outputs are still missing.
